# Preparing data for TTS transformer

### Tokenizing text

In [97]:
import re

from itertools import product
import numpy as np
import pandas as pd
import torch
import torchaudio
import umap
from datasets import Audio, load_dataset
from phonemizer import phonemize
from phonemizer.separator import Separator
from plotly import express as px
from sklearn_extra.cluster import KMedoids
from transformers import AutoProcessor, EncodecModel

In [2]:
def extract_phonemes(text):
    phonemes = phonemize(
        text,
        language="ru",
        backend="espeak",
        separator=Separator(phone="#", word=" | "),
        preserve_punctuation=False,
        with_stress=True,
        strip=True,
    )
    return phonemes

In [3]:
pattern = re.compile(r"\\u[0-9a-fA-F]{4}\d?")
corpus = ""
with open("../data/corpus.txt", encoding="utf-8") as f:
    for line in f.readlines():
        line = re.sub(r"[\u2020-\u203f]\d?", "", line)
        corpus += line

In [4]:
vocab = set(
    ["<s_ph>", "</s_ph>", "</s_mel>", "<s_mel>", "<UNK_ph>", "<UNK_mel>", "<space>"]
)
for word in extract_phonemes(corpus).split(" | "):
    vocab.update([phoneme for phoneme in word.split("#") if phoneme.isalpha()])
vocab = list(vocab)

In [5]:
len(vocab)

69

In [6]:
phon_to_idx = {phoneme: i for i, phoneme in enumerate(vocab)}
idx_to_phon = {i: phoneme for i, phoneme in enumerate(vocab)}


def tokenize(text):
    phonemes = extract_phonemes(text)
    tokens = [phon_to_idx["<s_ph>"]]
    for word in phonemes.split(" | "):
        for phoneme in word.split("#"):
            if phoneme in phon_to_idx.keys():
                tokens.append(phon_to_idx[phoneme])
            else:
                tokens.append(phon_to_idx["<UNK_ph>"])
        tokens.append(phon_to_idx["<space>"])
    tokens.append(phon_to_idx["</s_ph>"])
    return tokens


def untokenize(tokens):
    phonemes = []
    for token in tokens:
        if token in idx_to_phon.keys():
            phonemes.append(idx_to_phon[token])
        else:
            phonemes.append("<UNK_ph>")
    return phonemes

In [7]:
text = "Это обычное тестовое предложение."
tokens = tokenize(text)
new_text = untokenize(tokens)
print(tokens)
print(new_text)

[40, 32, 54, 16, 18, 16, 47, 48, 20, 19, 16, 56, 58, 18, 3, 32, 41, 54, 16, 59, 16, 56, 58, 18, 52, 2, 6, 63, 17, 16, 9, 32, 43, 6, 56, 58, 18, 26]
['<s_ph>', 'ˈɛ', 't', 'ʌ', '<space>', 'ʌ', 'b', 'ˈy', 'tʃʲ', 'n', 'ʌ', 'j', 'ɪ', '<space>', 'tʲ', 'ˈɛ', 's', 't', 'ʌ', 'v', 'ʌ', 'j', 'ɪ', '<space>', 'p', 'rʲ', 'i', 'd', 'ɭ', 'ʌ', 'ʒ', 'ˈɛ', 'nʲ', 'i', 'j', 'ɪ', '<space>', '</s_ph>']


### Tokenizing audio

In [8]:
librispeech_dummy = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
)
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")
librispeech_dummy = librispeech_dummy.cast_column(
    "audio", Audio(sampling_rate=processor.sampling_rate)
)
audio_sample = librispeech_dummy[-1]["audio"]["array"]
inputs = processor(
    raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt"
)
encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
audio_values = model.decode(
    encoder_outputs.audio_codes, encoder_outputs.audio_scales, inputs["padding_mask"]
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [182]:
encoder_outputs[0].shape

torch.Size([1, 1, 2, 336])

In [184]:
encoder_outputs[1]

[None]

In [11]:
second_codebook = model.quantizer.layers[1].codebook
embeddings = second_codebook.embed.numpy()
print("Embeddings shape:", embeddings.shape)

clusterized = KMedoids(n_clusters=78).fit(embeddings)
labels = clusterized.labels_
centroids = clusterized.cluster_centers_
print("Centroids shape:", centroids.shape)
combined_embeddings = np.vstack([embeddings, centroids])
print("Resulting shape:", combined_embeddings.shape)

reducer = umap.UMAP(n_components=3)
embeddings_3d = reducer.fit_transform(combined_embeddings)
print("Reduced shape:", embeddings_3d.shape)

Embeddings shape: (1024, 128)
Centroids shape: (78, 128)
Resulting shape: (1102, 128)
Reduced shape: (1102, 3)


In [12]:
df = pd.DataFrame.from_dict(
    [
        {
            "x": embeddings_3d[i, 0],
            "y": embeddings_3d[i, 1],
            "z": embeddings_3d[i, 2],
            "label": str(labels[i]),
            "symbol": "circle",
            "size_col": 4,
        }
        for i in range(1024)
    ]
    + [
        {
            "x": embeddings_3d[1024 + i, 0],
            "y": embeddings_3d[1024 + i, 1],
            "z": embeddings_3d[1024 + i, 2],
            "label": str(i),
            "symbol": "star",
            "size_col": 10,
        }
        for i in range(78)
    ]
)

In [13]:
df.head(3)

,x,y,z,label,symbol,size_col
0,3.537743,1.867749,3.135870,66,circle,4
1,4.516250,0.673584,2.309432,32,circle,4
2,4.160100,-0.091970,3.870343,44,circle,4


In [14]:
df.tail(3)

,x,y,z,label,symbol,size_col
1099,5.132276,-0.485494,4.380026,75,star,10
1100,3.667254,0.676741,4.064780,76,star,10
1101,1.625901,3.148265,3.199971,77,star,10


In [15]:
fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="label",
    size="size_col",
    symbol="symbol",
    width=1000,
    height=700,
)
fig.update_traces(
    marker=dict(opacity=1, line=dict(width=0, color="DarkSlateGrey")),
    selector=dict(mode="markers"),
)
fig.update_layout(
    title="<b>3D-projection of 2nd codebook embeddings</b>",
)
fig.show()

In [ ]:
second_cb_to_ind = {embedding: label for embedding, label in zip(range(1024), labels)}
ind_to_second_cb = {label: centroid for label, centroid in enumerate(centroids)}

encodec_to_tokens = {
    tuple(encodec_codes): token
    for token, encodec_codes in enumerate(list(product(range(1024), range(78))))
}
tokens_to_encodec = {
    token: tuple(encodec_codes)
    for token, encodec_codes in enumerate(list(product(range(1024), range(78))))
}

In [204]:
def process_audio(path_to_audio: str):
    waveform, orig_sr = torchaudio.load(path_to_audio)
    resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=24000)
    waveform = resampler(waveform)
    return waveform


def tokenize_waveform(waveform: np.array):
    model = EncodecModel.from_pretrained("facebook/encodec_24khz")
    processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")

    inputs = processor(
        raw_audio=waveform, sampling_rate=processor.sampling_rate, return_tensors="pt"
    )
    encoder_outputs = model.encode(
        inputs["input_values"], inputs["padding_mask"]
    ).audio_codes.squeeze()

    first_cb_codes = encoder_outputs[1]
    second_cb_codes = encoder_outputs[1]
    new_second_codes = torch.tensor(
        [second_cb_to_ind[int(code)] for code in second_cb_codes]
    )
    new_codes = torch.vstack([first_cb_codes, new_second_codes])

    tokenized = []
    for encoding in new_codes.transpose(1, 0):
        encoding = tuple([int(encoding[0]), int(encoding[1])])
        tokenized.append(encodec_to_tokens[encoding])
    return tokenized, inputs["padding_mask"]


def tokens_to_audio(audio_tokens: list, padding_mask: torch.tensor):
    model = EncodecModel.from_pretrained("facebook/encodec_24khz")
    encodec_tokens = []
    for token in audio_tokens:
        encodec_tokens.append(tokens_to_encodec[token])
    encodec_tokens = (
        torch.tensor(encodec_tokens).transpose(1, 0).unsqueeze(dim=0).unsqueeze(dim=0)
    )
    audio_values = model.decode(
        encodec_tokens,
        [None],
        padding_mask,
    )
    return audio_values

In [205]:
tokens, padding_mask = tokenize_waveform(audio_sample)
print(tokens)
print(padding_mask)

[66890, 66890, 42445, 40409, 73120, 40409, 71244, 71244, 40409, 71244, 40409, 71244, 40409, 40409, 42445, 33091, 33091, 40409, 33091, 33091, 33091, 42445, 33091, 33091, 33091, 40409, 33091, 40409, 40409, 73120, 42445, 33091, 40409, 23563, 40409, 33091, 33091, 40409, 33091, 33091, 71244, 66890, 66890, 28335, 65195, 52203, 13805, 50354, 19955, 44580, 27788, 75938, 71024, 45994, 71024, 71024, 17136, 43876, 32783, 31596, 78235, 65442, 63604, 78235, 16912, 63437, 2129, 51207, 51428, 71481, 73235, 42445, 65616, 71024, 75938, 39196, 2129, 5639, 62047, 54258, 43260, 29086, 54258, 68814, 23412, 43260, 16698, 54258, 75938, 78824, 79852, 76776, 78824, 49823, 7434, 60512, 62047, 12230, 61581, 42124, 56874, 60623, 67022, 40409, 54614, 73120, 75938, 60787, 75938, 75938, 27851, 75938, 56874, 8180, 76252, 11572, 57599, 9600, 55050, 954, 2013, 69634, 74669, 17339, 17339, 66890, 42242, 12524, 71244, 67927, 50896, 71244, 33091, 66890, 42242, 59168, 78824, 42242, 71024, 18920, 77187, 65442, 75607, 58758, 

In [206]:
audio = tokens_to_audio(tokens, padding_mask)
print(audio)

EncodecDecoderOutput(audio_values=tensor([[[0.0266, 0.0286, 0.0297,  ..., 0.3824, 0.3305, 0.2754]]],
       grad_fn=<ConvolutionBackward0>))
